# 🎬 Short-Drama-Factory 云端渲染引擎（Colab Pro+ ComfyUI）

**动静分离架构**：
- **Google Drive（`MyDrive/ComfyUI_Work/`）**：只存放 `input`、`output`、`workflows`、`loras`、`scripts`、`logs`。
- **Colab 高速本地盘（`/content/ComfyUI`）**：存放 ComfyUI 本体与几十 GB 大模型，每次开机通过 `HF_HUB_ENABLE_HF_TRANSFER=1` 满速直拉，关机自动清空。

**推荐执行顺序**：依次运行 Cell 1 → Cell 2 → Cell 3 → Cell 4，将 Cell 4 输出的 `https://xxxx.trycloudflare.com` 填入本地 `.env` 即可开始批量调度。

In [ ]:
#@title 1. 挂载 Google Drive 并定位维护脚本
import os
import subprocess
from pathlib import Path
from google.colab import drive

DRIVE_WORK_DIR = "/content/drive/MyDrive/ComfyUI_Work" #@param {type:"string"}
COMFY_DIR = "/content/ComfyUI" #@param {type:"string"}
# 可选：如果你把本仓库推到了 GitHub，填在这里即可每次开机自动拉取最新脚本；留空则直接读取 Drive/scripts/
GIT_REPO_URL = "https://github.com/ccxkai233/colab-short-drama.git" #@param {type:"string"}

drive.mount("/content/drive")

drive_work = Path(DRIVE_WORK_DIR)
for sub in ("input", "output", "workflows", "loras", "scripts", "logs"):
    (drive_work / sub).mkdir(parents=True, exist_ok=True)

if GIT_REPO_URL.strip():
    repo_dir = Path("/content/Short-Drama-Factory")
    if (repo_dir / ".git").exists():
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GIT_REPO_URL.strip(), str(repo_dir)], check=True)
    SCRIPTS_DIR = repo_dir / "colab"
    MANIFEST_PATH = repo_dir / "configs" / "models.yaml"
else:
    SCRIPTS_DIR = drive_work / "scripts"
    MANIFEST_PATH = SCRIPTS_DIR / "models.yaml"

os.environ["SDF_DRIVE_WORK_DIR"] = str(drive_work)
os.environ["SDF_COMFY_DIR"] = COMFY_DIR
os.environ["SDF_SCRIPTS_DIR"] = str(SCRIPTS_DIR)
os.environ["SDF_MANIFEST_PATH"] = str(MANIFEST_PATH)

required = ["setup_env.py", "patch_latentsync.py", "download_models.py", "idle_unassign.py"]
missing = [f for f in required if not (SCRIPTS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"在 {SCRIPTS_DIR} 中缺少脚本: {missing}\n"
        "请先将本地项目的 colab/*.py（以及可选的 configs/models.yaml）上传至 Google Drive 的 MyDrive/ComfyUI_Work/scripts/"
    )

print(f"✅ Drive 挂载成功 | 脚本目录: {SCRIPTS_DIR}")
print(f"✅ ComfyUI 本地高速目录: {COMFY_DIR}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
#@title 2. 读取 HF_TOKEN + 一键部署 ComfyUI、三大插件与 LatentSync 兼容补丁
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

FORCE_COMPAT_PATCH = False #@param {type:"boolean"}
UPDATE_EXISTING_REPOS = False #@param {type:"boolean"}

# 优先从 Colab Secrets 读取 HF_TOKEN
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or ""
    except Exception:
        hf_token = ""
if not hf_token:
    hf_token = getpass("未在 Colab Secrets 检测到 HF_TOKEN，请输入（回车跳过仅下公开模型）: ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN 已加载（不会明文打印）。")
else:
    print("⚠️ 未设置 HF_TOKEN，受限仓库（如 FLUX.1-dev）将无法下载。")

scripts_dir = Path(os.environ["SDF_SCRIPTS_DIR"])
comfy_dir = os.environ["SDF_COMFY_DIR"]
drive_work_dir = os.environ["SDF_DRIVE_WORK_DIR"]

def _run_stream(cmd, env=None):
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

# 2.1 部署环境与核心插件
setup_cmd = [
    sys.executable, str(scripts_dir / "setup_env.py"),
    "--drive-work-dir", drive_work_dir,
    "--comfy-dir", comfy_dir,
]
if UPDATE_EXISTING_REPOS:
    setup_cmd.append("--update")
_run_stream(setup_cmd)

# 2.2 探测并按需自动修补 LatentSync-Enhanced (torchaudio.save / torchvision.io.read_video)
patch_cmd = [
    sys.executable, str(scripts_dir / "patch_latentsync.py"),
    "--comfy-dir", comfy_dir,
    "--drive-work-dir", drive_work_dir,
]
if FORCE_COMPAT_PATCH:
    patch_cmd.append("--force")
_run_stream(patch_cmd)


In [ ]:
#@title 3. 按需高速拉取大模型（HF_HUB_ENABLE_HF_TRANSFER=1）
import os
import subprocess
import sys
from pathlib import Path

#@markdown ### 请按本次任务需要勾选要拉取的模型组（未勾选不会占用时间和本地磁盘）
DOWNLOAD_LATENTSYNC = True #@param {type:"boolean"}
DOWNLOAD_WAN22_FP8 = True #@param {type:"boolean"}
DOWNLOAD_WAN22_FP16 = False #@param {type:"boolean"}
DOWNLOAD_FLUX_DEV = True #@param {type:"boolean"}
DOWNLOAD_FLUX_KONTEXT = False #@param {type:"boolean"}
DOWNLOAD_FLUX_SCHNELL = False #@param {type:"boolean"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}

selected = []
if DOWNLOAD_LATENTSYNC:
    selected.append("latentsync")
if DOWNLOAD_WAN22_FP8:
    selected.append("wan22_fp8")
if DOWNLOAD_WAN22_FP16:
    selected.append("wan22_fp16")
if DOWNLOAD_FLUX_DEV:
    selected.append("flux_dev")
if DOWNLOAD_FLUX_KONTEXT:
    selected.append("flux_kontext")
if DOWNLOAD_FLUX_SCHNELL:
    selected.append("flux_schnell")

if not selected:
    print("未勾选任何模型组，跳过下载。")
else:
    scripts_dir = Path(os.environ["SDF_SCRIPTS_DIR"])
    manifest_path = Path(os.environ["SDF_MANIFEST_PATH"])
    cmd = [
        sys.executable, str(scripts_dir / "download_models.py"),
        "--groups", ",".join(selected),
        "--comfy-dir", os.environ["SDF_COMFY_DIR"],
        "--drive-work-dir", os.environ["SDF_DRIVE_WORK_DIR"],
    ]
    if manifest_path.exists():
        cmd.extend(["--manifest", str(manifest_path)])
    if FORCE_REDOWNLOAD:
        cmd.append("--force")

    env = os.environ.copy()
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


In [ ]:
#@title 4. 启动 ComfyUI + Cloudflared 穿透（8188）+ 可选任务完成自动释放算力
import importlib.util
import os
import re
import subprocess
import sys
import threading
import time
import urllib.request
from pathlib import Path

#@markdown ### 自动释放算力设置（防止跑完批量任务后空转扣计算单元）
ENABLE_AUTO_UNASSIGN = True #@param {type:"boolean"}
IDLE_MINUTES_AFTER_JOBS = 15 #@param {type:"number"}
MAX_NO_JOB_MINUTES = 60 #@param {type:"number"}
#@markdown ### 显存与启动参数（L4 24GB 建议留空默认；若爆显存可填 `--lowvram`）
EXTRA_COMFY_ARGS = "" #@param {type:"string"}

comfy_dir = Path(os.environ["SDF_COMFY_DIR"])
drive_work_dir = Path(os.environ["SDF_DRIVE_WORK_DIR"])
scripts_dir = Path(os.environ["SDF_SCRIPTS_DIR"])
logs_dir = drive_work_dir / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

comfy_log_path = logs_dir / "comfyui.log"
tunnel_url_path = logs_dir / "tunnel_url.txt"

# 1) 在后台线程启动 cloudflared 穿透 8188，并实时捕获公网 URL
tunnel_url_box = {"url": None}

def start_cloudflared():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    assert proc.stdout is not None
    for line in proc.stdout:
        m = pattern.search(line)
        if m and tunnel_url_box["url"] is None:
            url = m.group(0)
            tunnel_url_box["url"] = url
            tunnel_url_path.write_text(url + "\n", encoding="utf-8")
            print("\n" + "=" * 72, flush=True)
            print(f"🚀 ComfyUI 公网地址（填到本地 .env 的 COLAB_API_URL）:\n   {url}", flush=True)
            print(f"📝 已自动写入 Drive: {tunnel_url_path}", flush=True)
            print("=" * 72 + "\n", flush=True)

threading.Thread(target=start_cloudflared, daemon=True).start()

# 2) 可选：在当前 Colab Kernel 进程内启动空闲看门狗线程（确保 runtime.unassign() 可靠生效）
if ENABLE_AUTO_UNASSIGN:
    spec = importlib.util.spec_from_file_location("idle_unassign", scripts_dir / "idle_unassign.py")
    mod = importlib.util.module_from_spec(spec)
    assert spec and spec.loader
    spec.loader.exec_module(mod)
    threading.Thread(
        target=mod.run_watchdog,
        kwargs={
            "api_url": "http://127.0.0.1:8188",
            "idle_minutes": float(IDLE_MINUTES_AFTER_JOBS),
            "max_no_job_minutes": float(MAX_NO_JOB_MINUTES),
            "require_activity": True,
            "log_file": logs_dir / "unassign.log",
        },
        daemon=True,
    ).start()
    print(f"⏱️ 已开启自动释放看门狗：任务完成后空闲 {IDLE_MINUTES_AFTER_JOBS} 分钟自动 unassign。")

# 3) 启动 ComfyUI 主进程（同时回显并落盘日志到 Drive/logs/comfyui.log）
comfy_cmd = [
    sys.executable, "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--dont-print-server",
]
if EXTRA_COMFY_ARGS.strip():
    comfy_cmd.extend(EXTRA_COMFY_ARGS.strip().split())

print(f"正在启动 ComfyUI: {' '.join(comfy_cmd)}", flush=True)
with comfy_log_path.open("w", encoding="utf-8") as log_fh:
    proc = subprocess.Popen(
        comfy_cmd,
        cwd=str(comfy_dir),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    try:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            log_fh.write(line)
            log_fh.flush()
    except KeyboardInterrupt:
        print("\n收到手动中断，正在停止 ComfyUI...")
        proc.terminate()
        proc.wait(timeout=15)


In [ ]:
#@title 5. （手动）立即释放 Colab 算力
CONFIRM_UNASSIGN_NOW = False #@param {type:"boolean"}

if CONFIRM_UNASSIGN_NOW:
    from google.colab import runtime
    print("正在释放当前 Colab 运行时...")
    runtime.unassign()
else:
    print("未勾选 CONFIRM_UNASSIGN_NOW，已取消操作。")
